In [ ]:

import os
import glob
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import gc

In [ ]:

import os
import glob
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# =========================================================
# 1. 路径设置
# =========================================================

main_folder = r"E:\SHARE\E10006\DATAFILE\diagnosis"
output_folder = r"E:\SHARE\E10006\PARQUET"

os.makedirs(output_folder, exist_ok=True)

# =========================================================
# 2. 获取所有 dataset 子文件夹
# =========================================================

dataset_folders = sorted([
    folder
    for folder in os.listdir(main_folder)
    if os.path.isdir(os.path.join(main_folder, folder))
])

print("Dataset amount:", len(dataset_folders))

# =========================================================
# 3. 逐个 dataset 处理
# =========================================================

for dataset_name in dataset_folders:

    dataset_path = os.path.join(main_folder, dataset_name)

    gz_files = sorted(
        glob.glob(os.path.join(dataset_path, "*.gz"))
    )

    if len(gz_files) == 0:
        print(f"Skip {dataset_name}: no gz files found")
        continue

    output_file = os.path.join(
        output_folder,
        f"{dataset_name}.parquet"
    )

    # 临时文件：防止失败后保留不完整的正式 Parquet
    temp_file = output_file + ".incomplete"

    if os.path.exists(output_file):
        print(f"Skip {dataset_name}: Parquet already exists")
        continue

    if os.path.exists(temp_file):
        os.remove(temp_file)

    print("\n======================================")
    print("Dataset:", dataset_name)
    print("GZ file amount:", len(gz_files))
    print("======================================")

    writer = None
    expected_columns = None
    arrow_schema = None
    total_rows = 0

    try:

        for gz_file in gz_files:

            print("Reading:", os.path.basename(gz_file))

            for chunk in pd.read_csv(
                gz_file,
                compression="gzip",
                sep="\x01",
                chunksize=100000,
                dtype="string",
                low_memory=False,
                quoting=csv.QUOTE_NONE
            ):

                # 第一个 chunk 确定统一列名和 schema
                if writer is None:

                    expected_columns = list(chunk.columns)

                    arrow_schema = pa.schema([
                        pa.field(col, pa.string())
                        for col in expected_columns
                    ])

                    writer = pq.ParquetWriter(
                        temp_file,
                        arrow_schema,
                        compression="zstd"
                    )

                # 检查每一个 chunk 的列名和顺序
                if list(chunk.columns) != expected_columns:
                    raise ValueError(
                        f"Column mismatch in {gz_file}"
                    )

                # 强制所有 chunk 使用同一 Arrow schema
                table = pa.Table.from_pandas(
                    chunk,
                    schema=arrow_schema,
                    preserve_index=False
                )

                writer.write_table(table)

                total_rows += len(chunk)

                del chunk
                del table

            print("Cumulative rows:", f"{total_rows:,}")

            gc.collect()

        # =================================================
        # 4. 当前 dataset 成功写入后关闭并重命名
        # =================================================

        if writer is not None:

            writer.close()
            writer = None

            os.replace(temp_file, output_file)

            size_mb = os.path.getsize(output_file) / 1024**2

            # 检查 Parquet 元数据中的行数
            parquet_rows = pq.read_metadata(output_file).num_rows

            print("\nCompleted:", dataset_name)
            print("Total rows:", f"{total_rows:,}")
            print("Parquet rows:", f"{parquet_rows:,}")
            print("Parquet size:", round(size_mb, 2), "MB")

            if parquet_rows != total_rows:
                raise ValueError("Parquet row count mismatch")

        else:
            print("No data written:", dataset_name)

    except Exception as e:

        print("\nFAILED:", dataset_name)
        print("Error:", e)
        raise

    finally:

        if writer is not None:
            writer.close()

        # 临时文件不会被误认为正式完成的数据
        if os.path.exists(temp_file):
            os.remove(temp_file)

        gc.collect()


print("\nAll dataset processing completed!")


In [ ]:
# 处理outpatient data
# 1. 合并patient_data，by pseudo_patient_key

import os
import duckdb
import pyarrow.parquet as pq

# =========================================================
# 1. 路径设置
# =========================================================

parquet_folder = r"E:\SHARE\E10006\PARQUET"
output_folder = r"E:\SHARE\E10006\ANALYTIC"

os.makedirs(output_folder, exist_ok=True)

opas_file = os.path.join(
    parquet_folder, "opas_case_data.parquet"
)

patient_file = os.path.join(
    parquet_folder, "patient_data.parquet"
)

output_file = os.path.join(
    output_folder, "outpatient_D01.parquet"
)

temp_file = output_file + ".incomplete.parquet"

for f in [opas_file, patient_file]:
    if not os.path.isfile(f):
        raise FileNotFoundError(f)

if os.path.exists(output_file):
    raise FileExistsError(
        f"Output already exists: {output_file}"
    )

if os.path.exists(temp_file):
    os.remove(temp_file)


# =========================================================
# 2. 检查字段，并自动给 D01 变量加前缀
# =========================================================

opas_cols = pq.read_schema(opas_file).names
patient_cols = pq.read_schema(patient_file).names

join_key = "pseudo_patient_key"

if join_key not in opas_cols or join_key not in patient_cols:
    raise ValueError("Missing pseudo_patient_key")

def quote_col(name):
    return '"' + name.replace('"', '""') + '"'

patient_select = ",\n".join([
    f'p.{quote_col(col)} AS {quote_col("D01_" + col)}'
    for col in patient_cols
    if col != join_key
])

print("D08 columns:", len(opas_cols))
print("D01 columns:", len(patient_cols))


# =========================================================
# 3. 创建 DuckDB 连接
# =========================================================

con = duckdb.connect()

# 临时磁盘空间，用于大型 JOIN
temp_folder = os.path.join(
    output_folder, "duckdb_temp"
)
os.makedirs(temp_folder, exist_ok=True)

temp_sql = temp_folder.replace("\\", "/").replace("'", "''")

con.execute(
    f"SET temp_directory = '{temp_sql}'"
)

# 统一 SQL 文件路径
opas_sql = opas_file.replace("\\", "/").replace("'", "''")
patient_sql = patient_file.replace("\\", "/").replace("'", "''")
output_sql = temp_file.replace("\\", "/").replace("'", "''")


# =========================================================
# 4. 检查 D01 的 patient key 是否唯一
# =========================================================

check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT pseudo_patient_key) AS unique_patients,
    COUNT(*) FILTER (
        WHERE pseudo_patient_key IS NULL
    ) AS missing_keys
FROM read_parquet('{patient_sql}')
""").fetchone()

total, unique, missing = check

print("\nD01 rows:", total)
print("Unique patients:", unique)
print("Missing patient keys:", missing)

# 若同一患者有多行，会导致 JOIN 后行数增加
if total - missing != unique:
    raise ValueError(
        "D01 contains duplicated patient keys. "
        "Please resolve duplicates before merging."
    )


# =========================================================
# 5. LEFT JOIN
# =========================================================

sql = f"""
COPY (

    SELECT

        o.*,

        {patient_select}

    FROM read_parquet('{opas_sql}') AS o

    LEFT JOIN read_parquet('{patient_sql}') AS p

        ON o.pseudo_patient_key = p.pseudo_patient_key

)
TO '{output_sql}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
"""

print("\nStarting LEFT JOIN...")

try:

    con.execute(sql)

    # =====================================================
    # 6. 检查合并前后行数
    # =====================================================

    original_rows = pq.read_metadata(opas_file).num_rows
    merged_rows = pq.read_metadata(temp_file).num_rows

    print("Original D08 rows:", f"{original_rows:,}")
    print("Merged rows:", f"{merged_rows:,}")

    if original_rows != merged_rows:
        raise ValueError(
            "Row count changed after LEFT JOIN!"
        )

    # 检查成功后，正式保存
    os.replace(temp_file, output_file)

    size_gb = os.path.getsize(output_file) / 1024**3

    print("\nMerge completed!")
    print("Output:", output_file)
    print("Size:", round(size_gb, 2), "GB")

finally:
    con.close()


In [ ]:
#2.0. pre合并dmcs_data:先检查三个变量的匹配率
import duckdb

con = duckdb.connect()

result = con.execute("""
WITH sample AS (
    SELECT
        pseudo_record_key,
        pseudo_episode_key,
        pseudo_patient_key

    FROM 'E:/SHARE/E10006/ANALYTIC/outpatient_analysis.parquet'

    LIMIT 10000
)

SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE EXISTS (
            SELECT 1
            FROM 'E:/SHARE/E10006/PARQUET/dmcs_data.parquet' d

            WHERE sample.pseudo_record_key = d.pseudo_record_key
              AND sample.pseudo_episode_key = d.pseudo_episode_key
              AND sample.pseudo_patient_key = d.pseudo_patient_key
        )
    ) AS matched_rows

FROM sample
""").fetchdf()

print(result)

con.close()

In [ ]:
#2. 合并dmcs_data， by pseudo_record_key+pseudo_episode_key+pseudo_patient_key
import os
import duckdb
import pyarrow.parquet as pq

# =========================================================
# 1. 路径设置
# =========================================================

parquet_folder = r"E:\SHARE\E10006\PARQUET"
output_folder = r"E:\SHARE\E10006\ANALYTIC"

os.makedirs(output_folder, exist_ok=True)

# D08 + D01
base_file = os.path.join(
    output_folder,
    "outpatient_D01.parquet"
)

# D02
d02_file = os.path.join(
    parquet_folder,
    "dmcs_data.parquet"
)

# 新的输出文件
output_file = os.path.join(
    output_folder,
    "outpatient_D01_D02.parquet"
)

temp_file = output_file + ".incomplete.parquet"

# =========================================================
# 2. 检查文件是否存在
# =========================================================

for f in [base_file, d02_file]:
    if not os.path.isfile(f):
        raise FileNotFoundError(f)

if os.path.exists(output_file):
    raise FileExistsError(
        f"Output already exists: {output_file}"
    )

# =========================================================
# 3. 设置三个匹配变量
# =========================================================

join_keys = [
    "pseudo_record_key",
    "pseudo_episode_key",
    "pseudo_patient_key"
]

base_cols = pq.read_schema(base_file).names
d02_cols = pq.read_schema(d02_file).names

for key in join_keys:
    if key not in base_cols or key not in d02_cols:
        raise ValueError(f"Missing join key: {key}")

# 给 D02 的非匹配变量添加 D02_ 前缀
def quote_col(name):
    return '"' + name.replace('"', '""') + '"'

d02_select = ",\n".join([
    f'd.{quote_col(col)} AS {quote_col("D02_" + col)}'
    for col in d02_cols
    if col not in join_keys
])

print("Base columns:", len(base_cols))
print("D02 columns:", len(d02_cols))

# =========================================================
# 4. 创建 DuckDB 连接
# =========================================================

con = duckdb.connect()

temp_folder = os.path.join(
    output_folder, "duckdb_temp"
)

os.makedirs(temp_folder, exist_ok=True)

temp_sql = temp_folder.replace("\\", "/").replace("'", "''")
base_sql = base_file.replace("\\", "/").replace("'", "''")
d02_sql = d02_file.replace("\\", "/").replace("'", "''")
output_sql = temp_file.replace("\\", "/").replace("'", "''")

con.execute(f"SET temp_directory = '{temp_sql}'")

# =========================================================
# 5. 检查 D02 的三个匹配键是否有重复
# =========================================================

duplicates = con.execute(f"""
SELECT
    pseudo_record_key,
    pseudo_episode_key,
    pseudo_patient_key,
    COUNT(*) AS n

FROM read_parquet('{d02_sql}')

WHERE pseudo_record_key IS NOT NULL
  AND pseudo_episode_key IS NOT NULL
  AND pseudo_patient_key IS NOT NULL

GROUP BY
    pseudo_record_key,
    pseudo_episode_key,
    pseudo_patient_key

HAVING COUNT(*) > 1

LIMIT 5
""").fetchdf()

if not duplicates.empty:
    print(duplicates)
    con.close()
    raise ValueError(
        "Duplicated join keys found in D02!"
    )

print("D02 join keys: no duplicates found")

# =========================================================
# 6. LEFT JOIN
# =========================================================

sql = f"""
COPY (

    SELECT

        b.*,

        {d02_select}

    FROM read_parquet('{base_sql}') AS b

    LEFT JOIN read_parquet('{d02_sql}') AS d

        ON b.pseudo_record_key =
           d.pseudo_record_key

       AND b.pseudo_episode_key =
           d.pseudo_episode_key

       AND b.pseudo_patient_key =
           d.pseudo_patient_key

)
TO '{output_sql}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
"""

print("\nStarting D02 LEFT JOIN...")

try:

    if os.path.exists(temp_file):
        os.remove(temp_file)

    con.execute(sql)

    # =====================================================
    # 7. 检查合并前后行数
    # =====================================================

    original_rows = pq.read_metadata(base_file).num_rows
    merged_rows = pq.read_metadata(temp_file).num_rows

    print("Original rows:", f"{original_rows:,}")
    print("Merged rows:", f"{merged_rows:,}")

    if original_rows != merged_rows:
        raise ValueError(
            "Row count changed after LEFT JOIN!"
        )

    # 所有检查通过后保存正式文件
    os.replace(temp_file, output_file)

    size_gb = os.path.getsize(output_file) / 1024**3

    print("\nMerge completed!")
    print("Output:", output_file)
    print("Size:", round(size_gb, 2), "GB")

finally:

    con.close()

    if os.path.exists(temp_file):
        os.remove(temp_file)
